# 10 機器學習 — 練習

用松柏護理之家退伍軍人症資料練習 sklearn 分類 pipeline。

In [ ]:
# Google Colab setup -- 若在本機執行可跳過此 cell
import sys
import os
if 'google.colab' in sys.modules:
    !git clone https://github.com/ancientsky/python4epi.git /content/python4epi 2>/dev/null || true
    os.chdir('/content/python4epi')
    !pip install -q -e .

In [ ]:
import pathlib

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.metrics import roc_auc_score, classification_report
from sklearn.inspection import permutation_importance

# -- CJK font setup (避免中文標籤顯示為方框) --
# 掃描系統字型目錄，顯式註冊 CJK 字型（比依賴快取更可靠）
for _font_dir in map(pathlib.Path, ["/usr/share/fonts", "/usr/local/share/fonts"]):
    if _font_dir.exists():
        for _fp in sorted(_font_dir.rglob("*")):
            if _fp.suffix.lower() in {".ttf", ".ttc", ".otf"} and (
                "CJK" in _fp.name or "WenQuanYi" in _fp.name or "wqy" in _fp.name
            ):
                try:
                    fm.fontManager.addfont(str(_fp))
                except Exception:
                    pass

plt.rcParams["font.sans-serif"] = [
    "Noto Sans CJK TC", "Noto Sans CJK SC", "Noto Sans CJK JP",
    "Noto Sans TC", "Microsoft JhengHei",
    "WenQuanYi Zen Hei", "SimHei", "Arial Unicode MS",
    "Heiti TC", "DejaVu Sans",
]
plt.rcParams["axes.unicode_minus"] = False
plt.style.use("ggplot")
plt.rcParams["figure.dpi"] = 150

df = pd.read_csv("data/synthetic/legionella_outbreak.csv")
df["infected"] = (df["clinical_severity"] != "not_ill").astype(int)
df["severe_outcome"] = ((df["hospitalized"] == 1) | (df["outcome"] == "dead")).astype(int)

## 題目 1：class_weight="balanced" 的效果

1. 建立 Pipeline（ColumnTransformer + LogisticRegression）
2. 分別用 `class_weight=None` 和 `class_weight="balanced"` 訓練
3. 用 5-fold CV AUC 比較兩者在 Task B（severe_outcome）上的表現
4. 哪一個比較好？為什麼？

In [ ]:
# TODO: 定義特徵和前處理
# TODO: Pipeline with class_weight=None
# TODO: Pipeline with class_weight="balanced"
# TODO: 5-fold CV AUC 比較

## 題目 2：Task B（重症預測）的特徵重要性

1. 用 Random Forest 對 Task B（severe_outcome）做 5-fold CV AUC
2. 用 Permutation Importance 找出 top 5 重要特徵
3. 畫出特徵重要性的水平條圖
4. 與 Task A（infected）的重要特徵排序有何不同？

In [ ]:
# TODO: Random Forest pipeline
# TODO: 5-fold CV AUC
# TODO: Permutation importance
# TODO: barh plot + top 5

## 題目 3（挑戰題）：三模型比較 + ROC 曲線

1. 建立三個 Pipeline：Logistic Regression、Random Forest、Gradient Boosting
2. 用 70/30 split 訓練和測試（`random_state=42`）
3. 計算各模型在 Task A 上的 test AUC
4. 畫出三條 ROC 曲線在同一張圖上
5. 哪個模型表現最好？280 筆資料的結論可靠嗎？

In [ ]:
# TODO: 三個 Pipeline
# TODO: train_test_split(test_size=0.3, random_state=42)
# TODO: 各模型 AUC
# TODO: ROC 曲線（from sklearn.metrics import roc_curve）
# TODO: 解讀

## 題目 4：COVID-19 重症預測（COVID-19 情境）

以人口學與共病資料建立 COVID-19 重症的二元分類模型。

1. 以 `age/male/diabetes/hypertension/vaccinated` 為特徵、`severe` 為標籤
2. 切分訓練/測試集，建立 `StandardScaler + LogisticRegression` 的 Pipeline
3. 在測試集計算 accuracy、ROC-AUC、混淆矩陣
4. 從標準化係數解讀各特徵方向（特別是疫苗）

In [ ]:
# COVID-19：以人口學與共病預測「重症」（二元分類）
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
rng = np.random.default_rng(1004)
n = 1200
age = rng.integers(20, 90, n)
male = rng.integers(0, 2, n)
diabetes = rng.integers(0, 2, n)
hypertension = rng.integers(0, 2, n)
vaccinated = rng.binomial(1, 0.6, n)
logit = (-6 + 0.06 * age + 0.4 * male + 0.7 * diabetes + 0.5 * hypertension - 1.2 * vaccinated)
severe = rng.binomial(1, 1 / (1 + np.exp(-logit)))
covid = pd.DataFrame({"age": age, "male": male, "diabetes": diabetes,
                      "hypertension": hypertension, "vaccinated": vaccinated, "severe": severe})
print(covid["severe"].value_counts(normalize=True).round(3).to_dict())

# TODO: 以 age/male/diabetes/hypertension/vaccinated 為特徵，severe 為標籤
# TODO: train_test_split（stratify=y, test_size=0.25, random_state=42）
# TODO: 建立 Pipeline(StandardScaler + LogisticRegression) 並 fit
# TODO: 在測試集算 accuracy、ROC-AUC、confusion_matrix
# TODO: 解讀哪些特徵讓重症風險上升、疫苗的方向

## 題目 5：重症登革熱 DHF 預測（登革熱情境）

預測登革熱是否進展為重症（DHF）。

1. 特徵 `age/secondary_infection/platelet/days_fever`，標籤 `dhf`
2. 用 `RandomForestClassifier` 訓練，計算測試集 ROC-AUC 與混淆矩陣
3. 檢視 `feature_importances_`，判斷最關鍵的風險因子（提示：ADE）

In [ ]:
# 登革熱：預測是否進展為重症登革熱 DHF（隨機森林）
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix
rng = np.random.default_rng(1005)
n = 1000
age = rng.integers(1, 80, n)
secondary_infection = rng.binomial(1, 0.45, n)   # 二次感染是 ADE 風險
platelet = rng.normal(180, 60, n).clip(20, 400)  # 血小板(千/uL)
days_fever = rng.integers(1, 8, n)
logit = (-2.5 + 1.6 * secondary_infection - 0.012 * platelet + 0.15 * days_fever + 0.01 * age)
dhf = rng.binomial(1, 1 / (1 + np.exp(-logit)))
dengue = pd.DataFrame({"age": age, "secondary_infection": secondary_infection,
                       "platelet": platelet.round(0), "days_fever": days_fever, "dhf": dhf})
print(f"DHF 重症比例：{dengue['dhf'].mean():.1%}")

# TODO: 特徵 = age/secondary_infection/platelet/days_fever，標籤 = dhf
# TODO: train_test_split 後用 RandomForestClassifier（n_estimators=200）
# TODO: 算測試集 ROC-AUC 與混淆矩陣
# TODO: 看 feature_importances_，判斷哪個因子最重要（提示：二次感染 ADE）

## 題目 6：結核病治療結果預測（結核情境）

預測結核病人治療是否成功，並比較兩個模型。

1. 特徵 `age/mdr/hiv/adherence`，標籤 `success`
2. 用 `cross_val_score(cv=5, scoring='roc_auc')` 比較 LogisticRegression 與 RandomForest
3. 解讀服藥順從度 `adherence` 對治療成功的影響

In [ ]:
# 結核病：預測治療結果（成功 vs 失敗/中斷），比較兩個模型
from sklearn.metrics import roc_auc_score
rng = np.random.default_rng(1006)
n = 900
age = rng.integers(18, 85, n)
mdr = rng.binomial(1, 0.15, n)          # 抗藥性
hiv = rng.binomial(1, 0.1, n)
adherence = rng.uniform(0.4, 1.0, n)    # 服藥順從度
logit = (2.0 - 1.8 * mdr - 1.2 * hiv + 3.0 * (adherence - 0.7) - 0.01 * age)
success = rng.binomial(1, 1 / (1 + np.exp(-logit)))
tb = pd.DataFrame({"age": age, "mdr": mdr, "hiv": hiv,
                   "adherence": adherence.round(2), "success": success})
print(f"治療成功率：{tb['success'].mean():.1%}")

# TODO: 特徵 = age/mdr/hiv/adherence，標籤 = success
# TODO: 用 cross_val_score（cv=5, scoring="roc_auc"）比較 LogisticRegression 與 RandomForest
# TODO: 印出兩模型的平均 AUC，判斷哪個較好
# TODO: 解讀 adherence（服藥順從度）對治療成功的影響

## 題目 7：流感住院預測與 ROC 曲線（流感情境）

預測流感病人是否住院。

1. 特徵 `age/chronic/vaccinated/onset_to_care`，標籤 `hospitalized`
2. 用 `GradientBoostingClassifier` 訓練並計算 ROC-AUC
3. 用 `roc_curve` 畫出 ROC 曲線，解讀疫苗的方向

In [ ]:
# 流感：預測住院（梯度提升樹 + ROC 曲線）
from sklearn.metrics import roc_auc_score
rng = np.random.default_rng(1007)
n = 1100
age = rng.integers(0, 95, n)
chronic = rng.binomial(1, 0.25, n)
vaccinated = rng.binomial(1, 0.5, n)
onset_to_care = rng.integers(0, 6, n)  # 發病到就醫天數
logit = (-3.5 + 0.05 * age + 1.0 * chronic - 0.8 * vaccinated + 0.25 * onset_to_care)
hosp = rng.binomial(1, 1 / (1 + np.exp(-logit)))
flu = pd.DataFrame({"age": age, "chronic": chronic, "vaccinated": vaccinated,
                    "onset_to_care": onset_to_care, "hospitalized": hosp})
print(f"住院比例：{flu['hospitalized'].mean():.1%}")

# TODO: 特徵 = age/chronic/vaccinated/onset_to_care，標籤 = hospitalized
# TODO: 用 GradientBoostingClassifier 訓練，算測試集 ROC-AUC
# TODO: 用 roc_curve 畫 ROC 曲線
# TODO: 解讀疫苗接種對住院風險的方向

## 題目 8（挑戰題）：敗血症 ICU 預後與特徵重要度（敗血症情境）

預測敗血症病人的院內死亡，並比較兩種特徵重要度。

1. 特徵 `age/lactate/sofa/wbc/comorbid`，標籤 `death`
2. 用 `RandomForestClassifier` 訓練，計算 ROC-AUC
3. 用 `permutation_importance`（測試集）排出特徵重要度
4. 解讀：哪些床邊指標最重要？`permutation_importance` 與 `feature_importances_` 差在哪、為何前者較可信？

In [ ]:
# 敗血症：ICU 預後預測 + 特徵重要度（挑戰題）
from sklearn.metrics import roc_auc_score
rng = np.random.default_rng(1008)
n = 1000
age = rng.integers(18, 95, n)
lactate = rng.normal(2.5, 1.5, n).clip(0.5, 12)   # 乳酸
sofa = rng.integers(0, 18, n)                       # SOFA 分數
wbc = rng.normal(12, 6, n).clip(1, 40)
comorbid = rng.integers(0, 4, n)
logit = (-4 + 0.03 * age + 0.45 * lactate + 0.25 * sofa + 0.3 * comorbid + 0.01 * wbc)
death = rng.binomial(1, 1 / (1 + np.exp(-logit)))
sepsis = pd.DataFrame({"age": age, "lactate": lactate.round(1), "sofa": sofa,
                       "wbc": wbc.round(1), "comorbid": comorbid, "death": death})
print(f"院內死亡比例：{sepsis['death'].mean():.1%}")

# TODO: 特徵 = age/lactate/sofa/wbc/comorbid，標籤 = death
# TODO: train_test_split 後用 RandomForestClassifier 訓練，算 ROC-AUC
# TODO: 用 permutation_importance（在測試集）排出特徵重要度
# TODO: 解讀——哪些床邊指標對死亡預測貢獻最大？permutation_importance 與
#        feature_importances_ 有何不同、為何前者較可信？